In [1]:
# cell 1:
import sys, os, json
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
import joblib

from src.utils import load_config, set_seed
from src.cross_dataset.alignment import build_alignment_report, rename_2018_to_2017_schema
from src.preprocessing.cross_dataset_cleaning import clean_target_dataset, encode_target_labels
from src.evaluation.metrics import evaluate_model, print_metrics, results_to_dataframe

cfg = load_config()
set_seed(cfg["project"]["random_state"])

# --- Load 2018 raw data ---
path_2018 = os.path.join(cfg["paths"]["cicids2018_dir"], "Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv")
df_2018_raw = pd.read_csv(path_2018, low_memory=False)
df_2018_raw.columns = [str(c).strip() for c in df_2018_raw.columns]

# --- Load selected features + build alignment ---
with open(os.path.join(cfg["paths"]["reports_dir"], "selected_features.json")) as f:
    selection_info = json.load(f)
selected_features_2017 = selection_info["selected_features"]

alignment_report = build_alignment_report(selected_features_2017)
X_2018_aligned = rename_2018_to_2017_schema(df_2018_raw, alignment_report)
y_2018_raw = df_2018_raw["Label"]

print("Aligned 2018 features:", X_2018_aligned.shape)

# --- Clean and encode ---
X_2018_clean, y_2018_clean_raw = clean_target_dataset(X_2018_aligned, y_2018_raw)
y_2018_binary = encode_target_labels(y_2018_clean_raw, benign_label="Benign")

print("\nFinal 2018 test set:", X_2018_clean.shape)
print("Binary label distribution:")
print(y_2018_binary.value_counts())
print(f"Class balance: {y_2018_binary.value_counts(normalize=True).round(4).to_dict()}")

Aligned 2018 features: (1048575, 20)
TARGET DATASET (2018) CLEANING REPORT
Rows at start:                1,048,575
Duplicate rows removed:       487,465
Infinite values found:        0
Rows removed (NaN/inf):       0
Final rows:                   561,110

Final 2018 test set: (561110, 20)
Binary label distribution:
_label_raw
0    360519
1    200591
Name: count, dtype: int64
Class balance: {0: 0.6425, 1: 0.3575}


In [2]:
#cell2:
models_dir = cfg["paths"]["models_dir"]
final_rf = joblib.load(os.path.join(models_dir, "final_model_random_forest.joblib"))

in_dataset_results = pd.read_csv(os.path.join(cfg["paths"]["metrics_dir"], "final_model_metrics.csv"))
print("In-dataset (2017) performance:")
print(in_dataset_results[["model", "accuracy", "f1", "roc_auc", "fpr", "fnr"]])

In-dataset (2017) performance:
                           model  accuracy        f1  roc_auc       fpr  \
0  Final RF (tuned, 20 features)   0.99991  0.999922      1.0  0.000105   

        fnr  
0  0.000078  


In [3]:
#cell3:
cross_metrics = evaluate_model(final_rf, X_2018_clean, y_2018_binary, model_name="Final RF - Cross-Dataset (2018)")
print_metrics(cross_metrics)

RESULTS: Final RF - Cross-Dataset (2018)
Accuracy   : 0.6425
Precision  : 0.0000
Recall     : 0.0000
F1-score   : 0.0000
ROC-AUC    : 0.6004
PR-AUC     : 0.4748
FPR        : 0.0000
FNR        : 1.0000
------------------------------------------------------------
Confusion Matrix:
              Predicted BENIGN   Predicted DDoS
Actual BENIGN       360,519                 0
Actual DDoS         200,591                 0
------------------------------------------------------------
Prediction time  : 3.1347 sec


In [4]:
#Add cell4:
import numpy as np

print("Fwd Packet Length Max - by class and dataset:\n")

print("2017 (training domain):")
X_train_selected_path = os.path.join(cfg["paths"]["processed_dir"], "X_train_selected.parquet")
y_train_path = os.path.join(cfg["paths"]["processed_dir"], "y_train.parquet")
X_train_selected = pd.read_parquet(X_train_selected_path)
y_train = pd.read_parquet(y_train_path)["label_binary"]

print(X_train_selected.groupby(y_train)["Fwd Packet Length Max"].describe()[["mean", "std", "min", "50%", "max"]])

print("\n2018 (target domain):")
print(X_2018_clean.groupby(y_2018_binary)["Fwd Packet Length Max"].describe()[["mean", "std", "min", "50%", "max"]])

Fwd Packet Length Max - by class and dataset:

2017 (training domain):
                     mean          std  min   50%      max
label_binary                                              
0             1259.101140  2714.276404  0.0  42.0  11680.0
1               14.897111     6.738057  6.0  20.0     20.0

2018 (target domain):
                  mean         std  min    50%     max
_label_raw                                            
0           930.116088   66.163847  0.0  935.0  2224.0
1           244.459098  118.522571  0.0  291.0   365.0
